In [ ]:
import cv2 as cv
import threading
from time import sleep
import ipywidgets as widgets
from IPython.display import display
import Arm_Lib


Arm = Arm_Lib.Arm_Device()
joints_0 = [90, 135, 20, 25, 90, 30]
Arm.Arm_serial_servo_write6_array(joints_0, 1500)

class PositionalPID:
    def __init__(self, P, I, D):
        self.Kp = P # 比例系数
        self.Ki = I # 积分系数
        self.Kd = D # 微分系数
        self.SystemOutput = 0.0  # 初始化系统的输出值
        self.ResultValueBack = 0.0 # 初始化上次系统的输出值
        self.PidOutput = 0.0 # 初始化PID控制器的输出值
        self.PIDErrADD = 0.0 # 初始化PID误差累加值
        self.ErrBack = 0.0 # 初始化上次的误差值
    # 设置PID控制器参数
    def SetStepSignal(self, StepSignal):
        Err = StepSignal - self.SystemOutput  # 计算当前偏差
        KpWork = self.Kp * Err  # 计算比例项
        KiWork = self.Ki * self.PIDErrADD  # 计算积分项
        KdWork = self.Kd * (Err - self.ErrBack) # 计算微分项
        self.PidOutput = KpWork + KiWork + KdWork # 计算PID输出
        self.PIDErrADD += Err # 更新PID误差累加值
        self.ErrBack = Err # 更新上次的误差值
    # 设置一阶惯性环节系统  其中InertiaTime为惯性时间常数, SampleTime为采样时间
    def SetInertiaTime(self, InertiaTime, SampleTime):
        # 根据惯性时间常数和PID输出计算当前系统输出
        self.SystemOutput = (InertiaTime * self.ResultValueBack + \
                             SampleTime * self.PidOutput) / (SampleTime + InertiaTime)
        # 更新上次系统的输出值
        self.ResultValueBack = self.SystemOutput

class face_follow:
    def __init__(self):
        self.Arm = Arm_Lib.Arm_Device()
        self.target_servox=90  # 机械臂1号舵机的初始角度
        self.target_servoy=45  # 用于3号和4号舵机的角度控制
        self.xservo_pid = PositionalPID(0.5, 0.2, 0.35) 
        self.yservo_pid = PositionalPID(0.5, 0.2, 0.35)
        # 创建opencv的联级分类器的实例
        self.faceDetect = cv.CascadeClassifier("haarcascade_frontalface_default.xml")

    # 对人脸进行过滤
    def face_filter(self, faces):
        if len(faces) == 0: return None
        # 找画面中面积最大的人脸
        max_face = max(faces, key=lambda face: face[2] * face[3])
        (x, y, w, h) = max_face
        # 设置人脸检测最小阈值
        if w < 10 or h < 10: return None
        return max_face
    
    # 人脸追踪
    def follow_function(self, img):
        # 统一输入图像的大小
        img = cv.resize(img, (640, 480))
        # 复制原始图像,避免处理过程中干扰
        img = img.copy()
        # 图像灰度处理
        gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
        # 检测人脸
        faces = self.faceDetect.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)
        if len(faces) != 0:
            face = self.face_filter(faces) # 过滤人脸，返回面积最大的人脸
            (x, y, w, h) = face
            # 在原彩图上绘制矩形和标签
            cv.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 4)
            cv.putText(img, 'Person', (280, 30), cv.FONT_HERSHEY_SIMPLEX, 0.8, (105, 105, 105), 2)
            # 计算框中心点的坐标
            point_x = x + w / 2
            point_y = y + h / 2
            
            #  如果1号舵机的角度不大于180且检测框中心的x坐标大于320，或者1号舵机的角度值不小于0且检测框中心的x坐标小于320，则进行PID控制
            if not (self.target_servox>=180 and point_x<=320 or self.target_servox<=0 and point_x>=320):
                self.xservo_pid.SystemOutput = point_x  # 将检测框中心的x坐标作为PID的初始系统输出
                self.xservo_pid.SetStepSignal(320) # 设置舵机在x轴方向上的目标位置为320，更新PID控制器参数
                self.xservo_pid.SetInertiaTime(0.01, 0.1) # 设置PID的惯性时间
                # 更新1号舵机的角度值
                target_valuex = int(1500 + self.xservo_pid.SystemOutput)
                self.target_servox = int((target_valuex - 500) / 10)
                # 设置移动限制
                if self.target_servox > 180:self.target_servox = 180
                if self.target_servox < 0: self.target_servox = 0
             #  如果1号舵机的角度不大于180且检测框中心的y坐标大于240，或者1号舵机的角度值不小于0且检测框中心的y坐标小于240，则进行PID控制       
            if not (self.target_servoy>=180 and point_y<=240 or self.target_servoy<=0 and point_y>=240):
                # 输入Y轴方向参数PID控制输入
                self.yservo_pid.SystemOutput = point_y
                self.yservo_pid.SetStepSignal(240)
                self.yservo_pid.SetInertiaTime(0.01, 0.1)
                target_valuey = int(1500 + self.yservo_pid.SystemOutput)
                self.target_servoy = int((target_valuey - 500) / 10) - 45
                # 设置移动限制
                if self.target_servoy > 360: self.target_servoy = 360
                if self.target_servoy < 0: self.target_servoy = 0
            # 调整机械臂各个舵机的旋转角度
            joints_0 = [self.target_servox, 135, self.target_servoy / 2, self.target_servoy / 2, 90, 30]
            self.Arm.Arm_serial_servo_write6_array(joints_0, 300)
        return img


follow = face_follow()

# 退出控件
# 指定“Exit”按钮的尺寸和对齐方式
button_layout = widgets.Layout(width='250px', height='50px', align_self='center')
output = widgets.Output()

exit_button = widgets.Button(description='Exit', button_style='danger', layout=button_layout)
# 图像控件
imgbox = widgets.Image(format='jpg', height=480, width=640, layout=widgets.Layout(align_self='center'))
# 空间布局
controls_box = widgets.VBox([imgbox, exit_button], layout=widgets.Layout(align_self='center'))

# 初始化模式
model = 'General'

def exit_button_Callback(value):
    global model
    model = 'Exit'
exit_button.on_click(exit_button_Callback)

def camera():
    global model
    # 打开摄像头
    capture = cv.VideoCapture(0)
    # 当摄像头正常打开的情况下循环执行
    while capture.isOpened():
        try:
            # 读取相机的每一帧
            _, img = capture.read()
            # 统一图像大小
            img = cv.resize(img, (640, 480))
            img = follow.follow_function(img)
            if model == 'Exit':
                cv.destroyAllWindows()
                capture.release()
                break
            imgbox.value = cv.imencode('.jpg', img)[1].tobytes()
        except KeyboardInterrupt:capture.release()

display(controls_box,output)
threading.Thread(target=camera, ).start()
